In [ ]:
# 10회차 — 급식실 negative 확대 (9곳 236장 → 11곳 502장)
# 변수는 negative 구성 하나. 화염 소재는 9회차 flamelib2(724종) 그대로.
# 사전 등록: docs/PREREGISTER_R10.md
print('round10 notebook v1')
!nvidia-smi -L
!pip -q install ultralytics==8.3.* kagglehub


In [ ]:
# [1] 업로드 — 아래 9개
#   kitchen-fire-poc.zip
#   assets_1_bases.zip · assets_3_negsrc.zip · assets_4_weights.zip
#   flamelib2_a.zip · flamelib2_b.zip
#   round10_negsrc.zip        (신규 급식실 266장)
#   eval_F_fire.zip · eval_F_nofire.zip · eval_nonhyeon.zip
import zipfile, os, glob, shutil
from google.colab import files
up = files.upload()
os.makedirs('/content/work', exist_ok=True)
for n in up:
    zipfile.ZipFile(n).extractall('/content/work')
os.chdir('/content/work')

# 기존 negsrc + 신규 → 하나로 합침
n_old = len(glob.glob('assets/negsrc/*.jpg'))
for p in glob.glob('assets/negsrc_new/*.jpg'):
    shutil.copy(p, 'assets/negsrc/')
n_all = len(glob.glob('assets/negsrc/*.jpg'))
print(f'급식실 negative  기존 {n_old} + 신규 {n_all - n_old} = {n_all}장')
print('소재 flamelib2 ', len(glob.glob('assets/flamelib2/*.webp')))
print('평가군 F 화염  ', len(glob.glob('eval_F/F_fire/*.jpg')))
print('평가군 F 화염없음', len(glob.glob('eval_F/F_nofire/*.jpg')))
print('논현중(평가 전용)', len(glob.glob('eval_nonhyeon/*.jpg')))
assert n_all == 502, f'negative 502장이 아닙니다 ({n_all}) — zip 두 개를 모두 올렸는지 확인'
assert len(glob.glob('assets/flamelib2/*.webp')) == 724, '소재 724종이 아닙니다'


In [ ]:
# [2] D-Fire
import kagglehub
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DFIRE)


In [ ]:
# [3] 평가셋 A·C + 학습용 배경 (5회차와 동일 절차)
!python scripts/dfire_eval_set.py --dfire "$DFIRE" --out eval


In [ ]:
# [4] 합성 — 9회차와 동일, negative만 늘어남
!python scripts/synthesize.py --assets assets --flamelib flamelib2 --out ds \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600 --haze-prob 0.5
!cat ds/data.yaml


In [ ]:
# [5] 학습 — imgsz 640, 배치 16 (약 45분)
!yolo detect train model=yolov8s.pt data=ds/data.yaml epochs=60 imgsz=640 \
    batch=16 project=/content/runs name=r10 exist_ok=False


In [ ]:
# [6] 주 지표 — 논현중(학습 미사용 급식실) 오탐률
import glob, os, math, numpy as np, cv2
from ultralytics import YOLO
CONF = 0.10
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
W = {'5회차': 'assets/weights/round5_best.pt', '9회차': 'round9_best.pt', '10회차': best}
W = {k: v for k, v in W.items() if os.path.exists(v)}

SETS = {
    '논현중(미사용)': sorted(glob.glob('eval_nonhyeon/*.jpg')),
    '개원중 B': sorted(glob.glob('assets/eval_neg/*.jpg')),
    '영동중(학습)': sorted(glob.glob('assets/negsrc/yeongdong_*.jpg')),
    '원촌중(학습)': sorted(glob.glob('assets/negsrc/wonchon_*.jpg')),
}

def scan(m, ps, batch=32):
    out = []
    for i in range(0, len(ps), batch):
        for r in m.predict(ps[i:i+batch], conf=0.03, verbose=False):
            out.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    return np.array(out)

fp = {}
for tag, w in W.items():
    m = YOLO(w)
    fp[tag] = {k: scan(m, v) for k, v in SETS.items()}

print('=' * 70)
print('정상 조리 오탐률 (conf 0.10) — 전부 화재 없음')
hdr = ''.join(f'{k:>12s}' for k in W)
print(f"{'평가셋':22s}{hdr}")
for s in SETS:
    row = ''.join(f'{(fp[t][s] >= CONF).mean()*100:11.1f}%' for t in W)
    print(f'{s:22s}{row}   (n={len(SETS[s])})')

if '9회차' in fp and '10회차' in fp:
    d = ((fp['10회차']['논현중(미사용)'] >= CONF).mean()
         - (fp['9회차']['논현중(미사용)'] >= CONF).mean()) * 100
    print(f'\n논현중 변화 {d:+.1f}%p  →  ' +
          ('채택' if d <= -30 else '부분' if d <= -10 else '기각') + ' (사전 등록 기준)')


In [ ]:
# [7] 기각 조건 — 평가군 F 인식률이 무너지지 않았는지
FIRE = 0
P = sorted(glob.glob('eval_F/F_fire/*.jpg'))
N = sorted(glob.glob('eval_F/F_nofire/*.jpg'))
TAGS = sorted({os.path.basename(q).split('_')[0] for q in P + N})

def scanF(m, ps, batch=32):
    out = []
    for i in range(0, len(ps), batch):
        for r in m.predict(ps[i:i+batch], conf=0.03, verbose=False):
            f = 0.0
            if len(r.boxes):
                cl = r.boxes.cls.cpu().numpy().astype(int)
                cf = r.boxes.conf.cpu().numpy()
                if (cl == FIRE).any(): f = float(cf[cl == FIRE].max())
            out.append(f)
    return np.array(out)

res = {}
for tag, w in W.items():
    m = YOLO(w)
    zP, zN = scanF(m, P), scanF(m, N)
    recs, fprs = [], []
    for t in TAGS:
        ip = [i for i, q in enumerate(P) if os.path.basename(q).startswith(t)]
        inn = [i for i, q in enumerate(N) if os.path.basename(q).startswith(t)]
        if ip: recs.append((zP[ip] >= CONF).mean())
        if len(inn) >= 10: fprs.append((zN[inn] >= CONF).mean())   # 사전 등록 개정 반영
    res[tag] = (float(np.mean(recs)), float(np.mean(fprs)), zP, zN)

print('=' * 70)
print(f"{'':10s}{'F 인식률(macro)':>18s}{'F 오탐률(macro)':>18s}{'판별비 F':>12s}")
for t in W:
    r, f, _, _ = res[t]
    print(f'{t:10s}{r*100:17.1f}%{f*100:17.1f}%{r/f if f else float("inf"):12.2f}')

if '9회차' in res and '10회차' in res:
    dF = (res['10회차'][0] - res['9회차'][0]) * 100
    print(f'\nF 인식률 변화 {dF:+.1f}%p  →  ' +
          ('회귀 — 기각' if dF < -8 else '기준 이내'))
    # 판별비(F 인식률 ÷ 논현중 오탐률)
    for t in W:
        fpn = (fp[t]['논현중(미사용)'] >= CONF).mean()
        print(f'  {t}  판별비(F/논현중) {res[t][0]/fpn if fpn else float("inf"):.2f}')


In [ ]:
# [8] A·B·C 참고 기록
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('=' * 64); print('[5회차]'); print('=' * 64)
!python scripts/eval_gate.py --weights assets/weights/round5_best.pt \
    --eval-dir eval --cctv assets/eval_neg --conf 0.10
print('\n' + '=' * 64); print('[10회차]'); print('=' * 64)
!python scripts/eval_gate.py --weights "$best" --eval-dir eval --cctv assets/eval_neg --conf 0.10


In [ ]:
# [9] 논현중 오탐 사진 — 무엇이 남았는지가 11회차 설계의 근거
import cv2, numpy as np
from google.colab.patches import cv2_imshow
m10 = YOLO(best)
P9 = SETS['논현중(미사용)']
still = [P9[i] for i in np.where(fp['10회차']['논현중(미사용)'] >= CONF)[0]]
fixed = [P9[i] for i in np.where((fp['10회차']['논현중(미사용)'] < CONF) &
                                 (fp['9회차']['논현중(미사용)'] >= CONF))[0]] if '9회차' in fp else []
print(f'아직 오탐 {len(still)}장 · 9회차 대비 고쳐진 것 {len(fixed)}장')

def sheet(paths, title, n=12):
    if not paths: print(f'\n■ {title} 없음'); return
    tiles = []
    for p in paths[:n]:
        im = cv2.imread(p)
        for r in m10.predict(p, conf=CONF, verbose=False):
            for b, cf in zip(r.boxes.xyxy.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()):
                cv2.rectangle(im, (b[0], b[1]), (b[2], b[3]), (0, 0, 255), 2)
                cv2.putText(im, f'{cf:.2f}', (b[0], max(14, b[1]-4)),
                            cv2.FONT_HERSHEY_SIMPLEX, .6, (0, 0, 255), 2)
        tiles.append(cv2.resize(im, (400, 225)))
    while len(tiles) % 4: tiles.append(np.zeros((225, 400, 3), np.uint8))
    print(f'\n■ {title} ({len(paths)}장 중 앞 {min(n,len(paths))}장)')
    cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, len(tiles), 4)]))

sheet(still, '10회차가 아직 놓치지 못한 논현중 오탐')
sheet(fixed, '9회차 대비 고쳐진 것 (박스 없음이 정상)')


In [ ]:
# [10] 가중치 내려받기
from google.colab import files
files.download(max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime))
